In [1]:
import pandas as pd
import numpy as np
import holidays

# -----------------------------
# 1. LOAD DATA
# -----------------------------
path = "./data/"

lmp = pd.read_csv(path + "da_hrl_lmps (1).csv")
load_fcst = pd.read_csv(path + "load_frcstd_hist.csv")
gen = pd.read_csv(path + "gen_by_fuel.csv")
outages_raw = pd.read_csv(path + "frcstd_gen_outages.csv")

# -----------------------------
# 2. TARGET: whole PJM (pnode_id=1), day-ahead total LMP → total_lmp_da
# -----------------------------
lmp = lmp.loc[lmp["pnode_id"] == 1].copy()
lmp["datetime_beginning_ept"] = pd.to_datetime(lmp["datetime_beginning_ept"])
lmp = lmp.sort_values("datetime_beginning_ept")
lmp = lmp.drop_duplicates(subset=["datetime_beginning_ept"], keep="last")
lmp = lmp.set_index("datetime_beginning_ept")
lmp = lmp.rename(columns={"total_lmp_da": "price"})
lmp = lmp[["price"]]

# -----------------------------
# 3. LOAD FORECAST (7-day style file): forecast_hour_beginning_ept, forecast_load_mw
#    CSV has no datetime_beginning_ept; use RTO rows and latest evaluation per hour
# -----------------------------
load_fcst = load_fcst.loc[load_fcst["forecast_area"] == "RTO"].copy()
load_fcst["forecast_hour_beginning_ept"] = pd.to_datetime(load_fcst["forecast_hour_beginning_ept"])
load_fcst["evaluated_at_ept"] = pd.to_datetime(load_fcst["evaluated_at_ept"])
load_fcst = load_fcst.sort_values("evaluated_at_ept")
load_fcst = load_fcst.drop_duplicates(subset=["forecast_hour_beginning_ept"], keep="last")
load_fcst = load_fcst.set_index("forecast_hour_beginning_ept")
load_fcst = load_fcst.rename(columns={"forecast_load_mw": "load_forecast_mw"})
load_fcst = load_fcst[["load_forecast_mw"]]

# -----------------------------
# 4. GENERATION BY FUEL (wind/solar + full mix); fuel_type is title case in CSV
# -----------------------------
gen["datetime_beginning_ept"] = pd.to_datetime(gen["datetime_beginning_ept"])
gen = gen.sort_values("datetime_beginning_ept")
gen = gen.set_index("datetime_beginning_ept")

FUEL_MAP = {
    "Coal": "coal_mw",
    "Gas": "gas_mw",
    "Nuclear": "nuclear_mw",
    "Wind": "wind_mw",
    "Solar": "solar_mw",
    "Hydro": "hydro_mw",
    "Oil": "oil_mw",
    "Storage": "storage_mw",
    "Multiple Fuels": "multiple_fuels_mw",
    "Other Renewables": "other_renewables_mw",
}
gen_pivot = gen.pivot_table(
    index=gen.index,
    columns="fuel_type",
    values="mw",
    aggfunc="mean",
)
gen_pivot = gen_pivot.rename(columns=FUEL_MAP)
gen_pivot = gen_pivot[[c for c in FUEL_MAP.values() if c in gen_pivot.columns]]

# -----------------------------
# 5. FORECAST GENERATOR OUTAGES: forecast_gen_outage_mw_rto (daily; not hourly index)
#    Latest forecast_execution_date_ept per forecast_date; map each hour to that calendar day
# -----------------------------
outages = outages_raw.copy()
outages["forecast_date"] = pd.to_datetime(outages["forecast_date"])
outages["forecast_execution_date_ept"] = pd.to_datetime(outages["forecast_execution_date_ept"])
outages = outages.sort_values("forecast_execution_date_ept")
outages = outages.drop_duplicates(subset=["forecast_date"], keep="last")
outages = outages.set_index(outages["forecast_date"].dt.normalize())
outages = outages[["forecast_gen_outage_mw_rto"]]

# -----------------------------
# 6. MERGE
# -----------------------------
df = lmp.join(load_fcst, how="left")
df = df.join(gen_pivot, how="left")
df["forecast_gen_outage_mw_rto"] = outages["forecast_gen_outage_mw_rto"].reindex(
    df.index.normalize()
).to_numpy()

df = df.sort_index()
df = df.ffill(limit=3)

# -----------------------------
# 7. CALENDAR + CYCLICAL
# -----------------------------
df["hour"] = df.index.hour
df["day_of_week"] = df.index.dayofweek
df["month"] = df.index.month
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)


def get_season(month):
    if month in [12, 1, 2]:
        return 0
    if month in [3, 4, 5]:
        return 1
    if month in [6, 7, 8]:
        return 2
    return 3


df["season"] = df["month"].map(get_season)

us_holidays = holidays.US()
df["is_holiday"] = np.array([d.date() in us_holidays for d in df.index], dtype=int)

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

# -----------------------------
# 8. LAGS / DERIVED (lagged price per plan)
# -----------------------------
df["price_lag_1"] = df["price"].shift(1)
df["price_lag_24"] = df["price"].shift(24)
df["wind_mw"] = df["wind_mw"].fillna(0)
df["solar_mw"] = df["solar_mw"].fillna(0)
df["renewable_mw"] = df["wind_mw"] + df["solar_mw"]
df["price_roll_24"] = df["price"].rolling(24).mean()

df = df.dropna()

# -----------------------------
# 9. SAVE initial_data.csv
# -----------------------------
out = df.reset_index()
out.to_csv("initial_data.csv", index=False)
print(out.head())
print(out.shape)

/var/folders/vc/lspw4ql516j9z27l50xvy19h0000gn/T/ipykernel_41464/1678290850.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  lmp["datetime_beginning_ept"] = pd.to_datetime(lmp["datetime_beginning_ept"])
/var/folders/vc/lspw4ql516j9z27l50xvy19h0000gn/T/ipykernel_41464/1678290850.py:31: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  load_fcst["forecast_hour_beginning_ept"] = pd.to_datetime(load_fcst["forecast_hour_beginning_ept"])
/var/folders/vc/lspw4ql516j9z27l50xvy19h0000gn/T/ipykernel_41464/1678290850.py:42: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  gen["datetime_b

  datetime_beginning_ept      price  load_forecast_mw  coal_mw   gas_mw  \
0    2025-04-08 00:00:00  48.900998             83408  14591.0  33258.0   
1    2025-04-08 01:00:00  46.402496             82157  14779.0  32231.0   
2    2025-04-08 02:00:00  45.297246             81865  14776.0  31352.0   
3    2025-04-08 03:00:00  46.532548             82493  14694.0  31658.0   
4    2025-04-08 04:00:00  48.516346             81641  14732.0  32245.0   

   nuclear_mw  wind_mw  solar_mw  hydro_mw  oil_mw  ...  season  is_holiday  \
0     29350.0   4426.0      67.0     787.0   288.0  ...       1           0   
1     29364.0   4286.0      67.0     710.0   291.0  ...       1           0   
2     29369.0   4132.0      67.0     696.0   268.0  ...       1           0   
3     29371.0   3828.0      67.0     699.0   289.0  ...       1           0   
4     29359.0   3400.0      67.0     761.0   354.0  ...       1           0   

   hour_sin  hour_cos   dow_sin  dow_cos  price_lag_1  price_lag_24  \
0  

In [3]:
out.columns

Index(['datetime_beginning_ept', 'price', 'load_forecast_mw', 'coal_mw',
       'gas_mw', 'nuclear_mw', 'wind_mw', 'solar_mw', 'hydro_mw', 'oil_mw',
       'storage_mw', 'multiple_fuels_mw', 'other_renewables_mw',
       'forecast_gen_outage_mw_rto', 'hour', 'day_of_week', 'month',
       'is_weekend', 'season', 'is_holiday', 'hour_sin', 'hour_cos', 'dow_sin',
       'dow_cos', 'price_lag_1', 'price_lag_24', 'renewable_mw',
       'price_roll_24'],
      dtype='str')